In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
import numpy as np

c:\Users\davis\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [50]:
import re
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv('synthetic_logs.csv')
df.head()

,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert


In [3]:
df = df.drop(['complexity'], axis = 1)

In [4]:
df.head()

,timestamp,source,log_message,target_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status


In [5]:
df.source.unique()

<ArrowStringArray>
[      'ModernCRM', 'AnalyticsEngine',        'ModernHR',   'BillingSystem',
   'ThirdPartyAPI',       'LegacyCRM']
Length: 6, dtype: str

In [6]:
df.target_label.unique()

<ArrowStringArray>
[        'HTTP Status',      'Critical Error',      'Security Alert',
               'Error', 'System Notification',      'Resource Usage',
         'User Action',      'Workflow Error', 'Deprecation Warning']
Length: 9, dtype: str

In [7]:
#Load pre-trained sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2') #384 dims

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2161.96it/s]


In [8]:
#Generate embeddings for log messages
embeddings = model.encode(df['log_message'].tolist())

In [33]:
#clustering
dbscan = DBSCAN(eps=0.2, min_samples=4, metric='cosine')
clusters = dbscan.fit_predict(embeddings)

In [34]:
df['cluster'] = clusters
df.head()

,timestamp,source,log_message,target_label,cluster,regex_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,0,NaN
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,1,NaN
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,-1,NaN
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,0,NaN
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,0,NaN


In [35]:
df.cluster.unique()

array([ 0,  1, -1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 19, 13, 14,
       15, 16, 17, 18, 22, 20, 21, 27, 33, 32, 24, 23, 31, 25, 26, 28, 29,
       30, 37, 35, 34, 36], dtype=int64)

In [36]:
df[df.cluster == 1].head()

,timestamp,source,log_message,target_label,cluster,regex_label
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,1,NaN
10,8/9/2025 18:58,ModernCRM,Email server encountered a sending fault,Error,1,NaN
217,1/22/2025 5:45,BillingSystem,Mail service encountered a delivery glitch,Error,1,NaN
248,5/2/2025 23:04,ModernHR,Service disruption caused by email sending error,Critical Error,1,NaN
265,3/30/2025 23:53,ModernCRM,Email system had a problem sending emails,Error,1,NaN


In [37]:
def classify_with_regex(log_message):
    regex_patterns = {
        r"User User\d+ logged (in|out)." : "User Action",
        r"Backup (started|ended) at .*": "System Notification",
        r"Backup completed successfully": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully.": "System Notification",
        r"System reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }
    for pattern, label in regex_patterns.items():
        if re.search(pattern, log_message, re.IGNORECASE):
            return label
        return None

In [38]:
df['regex_label'] = df['log_message'].apply(classify_with_regex)
df

,timestamp,source,log_message,target_label,cluster,regex_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,0,NaN
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,1,NaN
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,-1,NaN
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,0,NaN
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,0,NaN
...,...,...,...,...,...,...
2405,2025-08-13 07:29:25,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,0,NaN
2406,1/11/2025 5:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,6,NaN
2407,2025-08-03 03:07:47,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,0,NaN
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,1,NaN


In [39]:
df[df.regex_label.isna()]

,timestamp,source,log_message,target_label,cluster,regex_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,0,NaN
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,1,NaN
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,-1,NaN
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,0,NaN
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,0,NaN
...,...,...,...,...,...,...
2405,2025-08-13 07:29:25,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,0,NaN
2406,1/11/2025 5:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,6,NaN
2407,2025-08-03 03:07:47,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,0,NaN
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,1,NaN


In [43]:
df[df.regex_label.notnull()]

,timestamp,source,log_message,target_label,cluster,regex_label
27,9/24/2025 19:57,ThirdPartyAPI,User User685 logged out.,User Action,10,User Action
57,9/14/2025 3:03,AnalyticsEngine,User User395 logged in.,User Action,10,User Action
85,3/13/2025 2:11,ModernHR,User User225 logged in.,User Action,10,User Action
88,3/8/2025 19:04,AnalyticsEngine,User User494 logged out.,User Action,10,User Action
126,11/22/2025 21:09,ThirdPartyAPI,User User900 logged in.,User Action,10,User Action
...,...,...,...,...,...,...
2207,10/4/2025 8:06,ModernCRM,User User495 logged in.,User Action,10,User Action
2263,2/27/2025 14:40,AnalyticsEngine,User User429 logged out.,User Action,10,User Action
2275,3/13/2025 17:17,AnalyticsEngine,User User755 logged out.,User Action,10,User Action
2323,12/1/2025 18:17,ThirdPartyAPI,User User882 logged out.,User Action,10,User Action


In [ ]:
df_non_regex = df[df['regex_label'].isnull()].copy()
df_non_regex

,timestamp,source,log_message,target_label,cluster,regex_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,0,NaN
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,1,NaN
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,-1,NaN
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,0,NaN
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,0,NaN
...,...,...,...,...,...,...
2405,2025-08-13 07:29:25,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,0,NaN
2406,1/11/2025 5:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,6,NaN
2407,2025-08-03 03:07:47,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,0,NaN
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,1,NaN


#BERT & Logistic Regression

In [46]:
print(df_non_regex['target_label'].value_counts() < 5)

target_label
HTTP Status            False
Security Alert         False
System Notification    False
Error                  False
Resource Usage         False
Critical Error         False
User Action            False
Workflow Error          True
Deprecation Warning     True
Name: count, dtype: bool


In [47]:
df_non_legacy = df_non_regex[df_non_regex.source!='LegacyCRM']
df_non_legacy.source.unique()

<ArrowStringArray>
['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem', 'ThirdPartyAPI']
Length: 5, dtype: str

In [48]:
#Generate embeddings
filtered_embeddings = model.encode(df_non_legacy['log_message'].tolist())
filtered_embeddings[:2]

array([[-1.02939621e-01,  3.35459746e-02, -2.20260657e-02,
         1.55103824e-03, -9.86923277e-03, -1.78956226e-01,
        -6.34409934e-02, -6.01761825e-02,  2.81108972e-02,
         5.99620566e-02, -1.72618572e-02,  1.43363792e-03,
        -1.49559990e-01,  3.15284869e-03, -5.66030890e-02,
         2.71685496e-02, -1.49890315e-02, -3.54037844e-02,
        -3.62936445e-02, -1.45410709e-02, -5.61493682e-03,
         8.75539035e-02,  4.55120690e-02,  2.50963699e-02,
         1.00187613e-02,  1.24266930e-02, -1.39923617e-01,
         7.68696293e-02,  3.14095132e-02, -4.15246096e-03,
         4.36902680e-02,  1.71250124e-02, -8.00951198e-02,
         5.74005730e-02,  1.89092048e-02,  8.55262280e-02,
         3.96399312e-02, -1.34371847e-01, -1.44363393e-03,
         3.06707132e-03,  1.76854044e-01,  4.44888510e-03,
        -1.69274844e-02,  2.24266667e-02, -4.35049534e-02,
         6.09027408e-03, -9.98167042e-03, -6.23972490e-02,
         1.07372822e-02, -6.04895130e-03, -7.14660957e-0

In [54]:
X = filtered_embeddings
y= df_non_legacy['target_label']

In [56]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)

                     precision    recall  f1-score   support

     Critical Error       0.94      1.00      0.97        46
              Error       1.00      0.94      0.97        52
        HTTP Status       1.00      1.00      1.00       294
     Resource Usage       1.00      1.00      1.00        50
     Security Alert       1.00      1.00      1.00       120
System Notification       1.00      1.00      1.00       115
        User Action       1.00      1.00      1.00        14

           accuracy                           1.00       691
          macro avg       0.99      0.99      0.99       691
       weighted avg       1.00      1.00      1.00       691



In [ ]:
import joblib
#joblib.dump(clf, 'models/log_classifier.joblib')

['models/log_classifier.joblib']